In [1]:
# emg_rf_baseline.py

import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit


# =========================
# CONFIG
# =========================
INPUT_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Merged_Features\Merged_Features.csv"
OUTPUT_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results"

RANDOM_STATE = 42
TEST_SIZE = 0.20

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# preferred/default names + fallback candidates
PARTICIPANT_CANDIDATES = ["Participant"]
SEX_CANDIDATES = ["Sex"]
TRIAL_CANDIDATES = ["Timeline", "Timestamp", "Trial Number"]
TARGET_CANDIDATES = ["Box Weight", "Box Mass"]
LOAD_TYPE_CANDIDATES = ["Load Type"]
HEIGHT_CANDIDATES = ["Lifting Height"]
DEPTH_CANDIDATES = ["Lifting Depth"]

# metadata to keep in outputs
OPTIONAL_METADATA_CANDIDATES = [
    "Participant",
    "Sex",
    "Timeline",
    "Timestamp",
    "Trial Number",
    "Box Weight",
    "Box Mass",
    "Load Type",
    "Lifting Height",
    "Lifting Depth",
    "Lowering Height",
    "Activity",
    "Age",
    "Stature",
    "Body Mass",
    "start_time",
    "end_time",
    "window_start_idx",
    "window_end_idx",
    "window_size",
    "step_size",
    "source_file",
]


# =========================
# HELPERS
# =========================
def ensure_dir(path: str) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"None of these columns were found: {candidates}")
    return ""


def normalize_sex_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    return str(x)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    if len(np.unique(y_true)) < 2:
        r2 = np.nan
    else:
        r2 = r2_score(y_true, y_pred)

    return {
        "R2": float(r2) if pd.notna(r2) else None,
        "MAE": float(mae),
        "RMSE": float(rmse),
    }


def build_trial_id(
    df: pd.DataFrame,
    participant_col: str,
    trial_col: str,
) -> pd.Series:
    return (
        df[participant_col].astype(str).fillna("NA")
        + "||"
        + df[trial_col].astype(str).fillna("NA")
    )


def select_feature_columns(
    df: pd.DataFrame,
    participant_col: str,
    sex_col: str,
    trial_col: str,
    target_col: str,
) -> List[str]:
    excluded = {
        participant_col,
        sex_col,
        trial_col,
        target_col,
        "trial_id",
        "split_group",
        "source_file",
        "start_time",
        "end_time",
        "window_start_idx",
        "window_end_idx",
        "window_size",
        "step_size",
        "Activity",
        "Load Type",
        "Lifting Height",
        "Lifting Depth",
        "Lowering Height",
        "Trial Number",
        "Age",
        "Stature",
        "Body Mass",
        "Timestamp",
        "Timeline",
        "Box Weight",
        "Box Mass",
    }

    emg_feature_cols = [c for c in df.columns if c.startswith("EMG_CH") and c not in excluded]
    if emg_feature_cols:
        return sorted(emg_feature_cols)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in excluded]
    if not feature_cols:
        raise ValueError("No usable feature columns found.")
    return sorted(feature_cols)


def aggregate_trial_predictions(
    df_test: pd.DataFrame,
    pred_col: str,
    trial_id_col: str,
    participant_col: str,
    sex_col: str,
    trial_col: str,
    target_col: str,
) -> pd.DataFrame:
    keep_cols = [trial_id_col, participant_col, sex_col, trial_col, target_col]
    keep_cols = [c for c in keep_cols if c in df_test.columns]

    agg_df = (
        df_test[keep_cols + [pred_col]]
        .groupby(trial_id_col, as_index=False)
        .agg(
            {
                participant_col: "first",
                sex_col: "first",
                trial_col: "first",
                target_col: "first",
                pred_col: "median",
            }
        )
        .rename(columns={pred_col: "y_pred_trial", target_col: "y_true_trial"})
    )

    agg_df[sex_col] = agg_df[sex_col].map(normalize_sex_value)
    return agg_df


def get_trial_counts_per_participant(
    df: pd.DataFrame,
    participant_col: str,
    trial_id_col: str,
) -> pd.DataFrame:
    out = (
        df[[participant_col, trial_id_col]]
        .drop_duplicates()
        .groupby(participant_col, as_index=False)
        .size()
        .rename(columns={"size": "n_trials"})
        .sort_values([participant_col])
        .reset_index(drop=True)
    )
    return out


def get_trial_counts_per_weight(
    df: pd.DataFrame,
    target_col: str,
    trial_id_col: str,
) -> pd.DataFrame:
    out = (
        df[[target_col, trial_id_col]]
        .drop_duplicates()
        .groupby(target_col, as_index=False)
        .size()
        .rename(columns={"size": "n_trials"})
        .sort_values([target_col])
        .reset_index(drop=True)
    )
    return out


def get_sex_specific_trial_metrics(
    trial_df: pd.DataFrame,
    sex_col: str,
) -> pd.DataFrame:
    rows = []
    for sex_value in ["Male", "Female"]:
        sub = trial_df[trial_df[sex_col] == sex_value].copy()
        if len(sub) == 0:
            rows.append(
                {
                    "Sex": sex_value,
                    "n_trials": 0,
                    "R2": None,
                    "MAE": None,
                    "RMSE": None,
                }
            )
            continue

        m = compute_metrics(sub["y_true_trial"].values, sub["y_pred_trial"].values)
        rows.append(
            {
                "Sex": sex_value,
                "n_trials": int(len(sub)),
                "R2": m["R2"],
                "MAE": m["MAE"],
                "RMSE": m["RMSE"],
            }
        )

    return pd.DataFrame(rows)


def save_json(obj: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# =========================
# MAIN PIPELINE
# =========================
def main():
    out_dir = ensure_dir(OUTPUT_DIR)

    df = pd.read_csv(INPUT_CSV)

    participant_col = resolve_column(df, PARTICIPANT_CANDIDATES, required=True)
    sex_col = resolve_column(df, SEX_CANDIDATES, required=True)
    trial_col = resolve_column(df, TRIAL_CANDIDATES, required=True)
    target_col = resolve_column(df, TARGET_CANDIDATES, required=True)

    df[sex_col] = df[sex_col].map(normalize_sex_value)
    df["trial_id"] = build_trial_id(df, participant_col=participant_col, trial_col=trial_col)

    metadata_cols = [c for c in OPTIONAL_METADATA_CANDIDATES if c in df.columns]
    feature_cols = select_feature_columns(
        df,
        participant_col=participant_col,
        sex_col=sex_col,
        trial_col=trial_col,
        target_col=target_col,
    )

    required_cols = [participant_col, sex_col, trial_col, target_col, "trial_id"] + feature_cols
    work_df = df[required_cols + [c for c in metadata_cols if c not in required_cols]].copy()

    work_df = work_df.dropna(subset=[participant_col, sex_col, trial_col, target_col]).copy()

    # feature rows with NaN/Inf are dropped
    X_all = work_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    valid_mask = X_all.notna().all(axis=1)
    work_df = work_df.loc[valid_mask].reset_index(drop=True)

    if len(work_df) == 0:
        raise ValueError("No valid rows remain after dropping NaN/Inf feature rows.")

    # split strategy
    unique_participants = work_df[participant_col].dropna().unique()
    if len(unique_participants) >= 2:
        split_groups = work_df[participant_col].astype(str).values
        split_strategy = "subject_level"
    else:
        split_groups = work_df["trial_id"].astype(str).values
        split_strategy = "trial_group_level"

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(work_df, groups=split_groups))

    train_df = work_df.iloc[train_idx].reset_index(drop=True)
    test_df = work_df.iloc[test_idx].reset_index(drop=True)

    # safety check: no trial leakage
    shared_trials = set(train_df["trial_id"]).intersection(set(test_df["trial_id"]))
    if len(shared_trials) > 0:
        raise RuntimeError("Leakage detected: some trials appear in both train and test.")

    X_train = train_df[feature_cols].values
    y_train = train_df[target_col].astype(float).values

    X_test = test_df[feature_cols].values
    y_test = test_df[target_col].astype(float).values

    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train, y_train)

    # window-level predictions
    test_df = test_df.copy()
    test_df["y_pred_window"] = model.predict(X_test)
    test_df["y_true_window"] = y_test

    window_metrics = compute_metrics(
        test_df["y_true_window"].values,
        test_df["y_pred_window"].values,
    )

    # trial-level aggregation by median
    trial_pred_df = aggregate_trial_predictions(
        df_test=test_df,
        pred_col="y_pred_window",
        trial_id_col="trial_id",
        participant_col=participant_col,
        sex_col=sex_col,
        trial_col=trial_col,
        target_col=target_col,
    )

    trial_metrics = compute_metrics(
        trial_pred_df["y_true_trial"].values,
        trial_pred_df["y_pred_trial"].values,
    )

    sex_metrics_df = get_sex_specific_trial_metrics(trial_pred_df, sex_col=sex_col)
    participant_trial_counts_df = get_trial_counts_per_participant(
        work_df, participant_col=participant_col, trial_id_col="trial_id"
    )
    weight_trial_counts_df = get_trial_counts_per_weight(
        work_df, target_col=target_col, trial_id_col="trial_id"
    )

    feature_importance_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": model.feature_importances_,
        }
    ).sort_values("importance", ascending=False).reset_index(drop=True)

    # save outputs
    summary = {
        "input_csv": INPUT_CSV,
        "output_dir": str(out_dir),
        "split_strategy": split_strategy,
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "n_total_windows": int(len(work_df)),
        "n_train_windows": int(len(train_df)),
        "n_test_windows": int(len(test_df)),
        "n_total_trials": int(work_df["trial_id"].nunique()),
        "n_train_trials": int(train_df["trial_id"].nunique()),
        "n_test_trials": int(test_df["trial_id"].nunique()),
        "participant_column": participant_col,
        "sex_column": sex_col,
        "trial_column": trial_col,
        "target_column": target_col,
        "n_features": int(len(feature_cols)),
        "window_level_metrics": window_metrics,
        "trial_level_metrics": trial_metrics,
    }

    keep_window_output_cols = [c for c in metadata_cols if c in test_df.columns] + [
        "trial_id",
        "y_true_window",
        "y_pred_window",
    ]
    test_df[keep_window_output_cols].to_csv(out_dir / "test_window_predictions.csv", index=False)
    trial_pred_df.to_csv(out_dir / "test_trial_predictions.csv", index=False)
    sex_metrics_df.to_csv(out_dir / "trial_metrics_by_sex.csv", index=False)
    participant_trial_counts_df.to_csv(out_dir / "trial_counts_per_participant.csv", index=False)
    weight_trial_counts_df.to_csv(out_dir / "trial_counts_per_weight.csv", index=False)
    feature_importance_df.to_csv(out_dir / "feature_importance.csv", index=False)
    save_json(summary, out_dir / "summary.json")

    # concise terminal output
    print("=== Random Forest EMG baseline complete ===")
    print(f"Split strategy: {split_strategy}")
    print(f"Train windows: {len(train_df)} | Test windows: {len(test_df)}")
    print(f"Train trials: {train_df['trial_id'].nunique()} | Test trials: {test_df['trial_id'].nunique()}")
    print()
    print("Window-level metrics")
    print(window_metrics)
    print()
    print("Trial-level metrics (main)")
    print(trial_metrics)
    print()
    print("Trial-level metrics by sex")
    print(sex_metrics_df.to_string(index=False))
    print()
    print("Top 20 feature importances")
    print(feature_importance_df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()

=== Random Forest EMG baseline complete ===
Split strategy: subject_level
Train windows: 641283 | Test windows: 160032
Train trials: 365 | Test trials: 94

Window-level metrics
{'R2': 0.09366268170151948, 'MAE': 2.2782628682596804, 'RMSE': 2.779288356772614}

Trial-level metrics (main)
{'R2': 0.14742482856412809, 'MAE': 2.1984326241134697, 'RMSE': 2.722625133617056}

Trial-level metrics by sex
   Sex  n_trials       R2     MAE     RMSE
  Male        59 0.120809 2.21791 2.758932
Female        35 0.191784 2.16560 2.660301

Top 20 feature importances
     feature  importance
 EMG_CH5_min    0.020642
EMG_CH5_mean    0.019849
EMG_CH5_iemg    0.019325
 EMG_CH5_mav    0.019130
 EMG_CH4_min    0.018919
 EMG_CH5_max    0.018883
 EMG_CH3_min    0.017907
 EMG_CH5_rms    0.017474
 EMG_CH7_min    0.017111
 EMG_CH6_min    0.017097
 EMG_CH7_max    0.016839
 EMG_CH6_max    0.016751
 EMG_CH8_max    0.016270
 EMG_CH4_mav    0.016069
 EMG_CH2_max    0.016063
 EMG_CH8_min    0.016043
EMG_CH4_mean    0.015

In [2]:
# analyze_trial_predictions.py

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================
# CONFIG
# =========================
TRIAL_PRED_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\test_trial_predictions.csv"
OUTPUT_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis"

TRUE_COL = "y_true_trial"
PRED_COL = "y_pred_trial"
SEX_COL = "Sex"


# =========================
# HELPERS
# =========================
def ensure_dir(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    return {
        "R2": float(r2) if pd.notna(r2) else None,
        "MAE": float(mae),
        "RMSE": float(rmse),
    }


def normalize_sex(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    return str(x)


# =========================
# MAIN
# =========================
def main():
    out_dir = ensure_dir(OUTPUT_DIR)
    df = pd.read_csv(TRIAL_PRED_CSV)

    df[SEX_COL] = df[SEX_COL].map(normalize_sex)

    df["error"] = df[PRED_COL] - df[TRUE_COL]
    df["abs_error"] = np.abs(df["error"])
    df["sq_error"] = df["error"] ** 2

    # overall metrics
    overall_metrics = compute_metrics(df[TRUE_COL], df[PRED_COL])
    pd.DataFrame([overall_metrics]).to_csv(out_dir / "overall_trial_metrics.csv", index=False)

    # per-weight error
    per_weight_rows = []
    for weight, g in df.groupby(TRUE_COL, dropna=False):
        m = compute_metrics(g[TRUE_COL], g[PRED_COL])
        per_weight_rows.append(
            {
                "weight_level": weight,
                "n_trials": int(len(g)),
                "mean_true": float(g[TRUE_COL].mean()),
                "mean_pred": float(g[PRED_COL].mean()),
                "mean_error": float(g["error"].mean()),
                "mean_abs_error": float(g["abs_error"].mean()),
                "median_abs_error": float(g["abs_error"].median()),
                "RMSE": m["RMSE"],
                "R2": m["R2"],
            }
        )

    per_weight_df = pd.DataFrame(per_weight_rows).sort_values("weight_level").reset_index(drop=True)
    per_weight_df.to_csv(out_dir / "per_weight_error.csv", index=False)

    # per-weight by sex
    per_weight_sex_rows = []
    for (weight, sex), g in df.groupby([TRUE_COL, SEX_COL], dropna=False):
        if len(g) == 0:
            continue
        m = compute_metrics(g[TRUE_COL], g[PRED_COL])
        per_weight_sex_rows.append(
            {
                "weight_level": weight,
                "Sex": sex,
                "n_trials": int(len(g)),
                "mean_pred": float(g[PRED_COL].mean()),
                "mean_error": float(g["error"].mean()),
                "mean_abs_error": float(g["abs_error"].mean()),
                "RMSE": m["RMSE"],
                "R2": m["R2"],
            }
        )

    per_weight_sex_df = (
        pd.DataFrame(per_weight_sex_rows)
        .sort_values(["weight_level", "Sex"])
        .reset_index(drop=True)
    )
    per_weight_sex_df.to_csv(out_dir / "per_weight_error_by_sex.csv", index=False)

    # scatter plot: true vs pred
    plt.figure(figsize=(6, 6))
    plt.scatter(df[TRUE_COL], df[PRED_COL], alpha=0.7)
    min_v = min(df[TRUE_COL].min(), df[PRED_COL].min())
    max_v = max(df[TRUE_COL].max(), df[PRED_COL].max())
    plt.plot([min_v, max_v], [min_v, max_v], linewidth=1)
    plt.xlabel("True Box Weight")
    plt.ylabel("Predicted Box Weight")
    plt.title("Trial-level Prediction vs True")
    plt.tight_layout()
    plt.savefig(out_dir / "trial_scatter_true_vs_pred.png", dpi=300, bbox_inches="tight")
    plt.close()

    # boxplot-like summary via matplotlib: absolute error by weight
    weight_levels = sorted(df[TRUE_COL].dropna().unique().tolist())
    box_data = [df.loc[df[TRUE_COL] == w, "abs_error"].values for w in weight_levels]

    plt.figure(figsize=(8, 5))
    plt.boxplot(box_data, labels=[str(w) for w in weight_levels])
    plt.xlabel("True Box Weight")
    plt.ylabel("Absolute Error")
    plt.title("Trial-level Absolute Error by Weight")
    plt.tight_layout()
    plt.savefig(out_dir / "trial_abs_error_by_weight_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()

    # mean abs error by weight
    plt.figure(figsize=(8, 5))
    plt.bar(per_weight_df["weight_level"].astype(str), per_weight_df["mean_abs_error"])
    plt.xlabel("True Box Weight")
    plt.ylabel("Mean Absolute Error")
    plt.title("Trial-level Mean Absolute Error by Weight")
    plt.tight_layout()
    plt.savefig(out_dir / "trial_mean_abs_error_by_weight_bar.png", dpi=300, bbox_inches="tight")
    plt.close()

    # rounded-weight confusion-style table
    df["pred_weight_rounded"] = df[PRED_COL].round().astype(int)
    confusion_like = pd.crosstab(df[TRUE_COL], df["pred_weight_rounded"])
    confusion_like.to_csv(out_dir / "rounded_weight_confusion_style.csv")

    print("Saved:")
    print(out_dir / "overall_trial_metrics.csv")
    print(out_dir / "per_weight_error.csv")
    print(out_dir / "per_weight_error_by_sex.csv")
    print(out_dir / "trial_scatter_true_vs_pred.png")
    print(out_dir / "trial_abs_error_by_weight_boxplot.png")
    print(out_dir / "trial_mean_abs_error_by_weight_bar.png")
    print(out_dir / "rounded_weight_confusion_style.csv")


if __name__ == "__main__":
    main()

Saved:
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\overall_trial_metrics.csv
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\per_weight_error.csv
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\per_weight_error_by_sex.csv
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\trial_scatter_true_vs_pred.png
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\trial_abs_error_by_weight_boxplot.png
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\trial_mean_abs_error_by_weight_bar.png
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\RF_Baseline_Results\Post_Analysis\rounded_weight_confusion_style.csv
